# Tests: `fastermodels.eval` (source `nbs/01_eval.ipynb`)

In [ ]:
from fastcore.test import *
import copy

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from fastermodels.eval import (PairedDelta, _mcnemar, agreement, correct_vector, macs, paired_delta,
                               params, peak_activation_bytes, predictions, wilson)

In [ ]:
# Wilson against a known table value: 40421 correct out of 50000
test_close(wilson(40421, 50000), (0.80495, 0.81185), eps=1e-4)

# an accuracy of 0 has a lower bound of exactly 0, and the interval stays inside [0, 1]
test_eq(wilson(0, 10)[0], 0.)
assert 0 < wilson(0, 10)[1] < 1
test_eq(wilson(10, 10)[1], 1.)

# a wider z gives a wider interval, more images give a narrower one
_lo95, _hi95 = wilson(90, 100)
_lo99, _hi99 = wilson(90, 100, z=2.576)
assert _lo99 < _lo95 and _hi99 > _hi95
assert (wilson(9000, 10000)[1] - wilson(9000, 10000)[0]) < (_hi95 - _lo95)

with ExceptionExpected(ValueError, regex='n > 0'): wilson(0, 0)

In [ ]:
_a = np.zeros(1000, dtype=bool); _a[:800] = True

# the same model against itself: no difference, no discordant pair
_same = paired_delta(_a, _a)
test_eq(type(_same), PairedDelta)
test_eq(_same.delta, 0.)
assert _same.lo <= 0 <= _same.hi
test_eq(_same.p_mcnemar, 1.)
test_eq(_same.n, 1000)
test_eq(_same.as_dict()['n'], 1000)

# ten more images right out of a thousand: +1.0 point, an interval clear of zero, and a small p
_b = _a.copy(); _b[800:810] = True
_better = paired_delta(_a, _b)
test_close(_better.delta, 1.0, eps=1e-9)
assert _better.lo > 0
assert _better.p_mcnemar < 0.01

# the sign follows b - a
test_close(paired_delta(_b, _a).delta, -1.0, eps=1e-9)

# same seed, same interval
test_eq(paired_delta(_a, _b).as_dict(), _better.as_dict())

# the exact McNemar p only reads the discordant counts
test_eq(_mcnemar(0, 0), 1.0)
test_close(_mcnemar(0, 10), 2 / 2 ** 10, eps=1e-12)
test_eq(_mcnemar(5, 5), 1.0)

with ExceptionExpected(ValueError, regex='same images'): paired_delta(_a, _a[:10])
with ExceptionExpected(ValueError, regex='empty'): paired_delta(np.array([], dtype=bool), np.array([], dtype=bool))

In [ ]:
torch.manual_seed(0)
_X, _y = torch.randn(20, 4), torch.tensor([0, 1] * 10)
_dl = DataLoader(TensorDataset(_X, _y), batch_size=8)
_model = nn.Linear(4, 2).eval()

# the harness agrees with the hand computation, image by image and in dataloader order
_hand = (_model(_X).argmax(1) == _y).numpy()
_cv = correct_vector(_model, _dl)
test_eq(_cv.dtype, np.dtype(bool))
test_eq(_cv.tolist(), _hand.tolist())
test_eq(predictions(_model, _dl).tolist(), _model(_X).argmax(1).tolist())

# a model that is right everywhere and one that is wrong everywhere
test_eq(correct_vector(lambda x: nn.functional.one_hot(torch.tensor([0, 1] * (len(x) // 2)), 2).float(), _dl).all(), True)

# agreement is a fraction over the same images
_p = predictions(_model, _dl)
test_eq(agreement(_p, _p), 1.0)
test_eq(agreement(_p, 1 - _p), 0.0)
test_close(agreement(np.array([0, 1, 2, 3]), np.array([0, 1, 9, 9])), 0.5, eps=1e-12)
with ExceptionExpected(ValueError, regex='same images'): agreement(_p, _p[:2])
with ExceptionExpected(ValueError, regex='empty'): agreement(np.array([]), np.array([]))

# the harness never touches the caller's model: it stays where it was, and in the mode it was in
_where = {n: p.device for n, p in _model.named_parameters()}
correct_vector(_model, _dl, device='cpu')
test_eq({n: p.device for n, p in _model.named_parameters()}, _where)
test_eq(_model.training, False)

In [ ]:
# logits that are not finite raise instead of scoring a silent 0 %
class _Nan(nn.Module):
    def forward(self, x): return torch.full((x.shape[0], 2), float('nan'))

with ExceptionExpected(ValueError, regex='not finite'): correct_vector(_Nan().eval(), _dl)
with ExceptionExpected(ValueError, regex='not finite'): predictions(_Nan().eval(), _dl)

# a model left in training mode raises: its BatchNorm would score the batch, not the model
with ExceptionExpected(ValueError, regex='training mode'): correct_vector(nn.Linear(4, 2).train(), _dl)
with ExceptionExpected(ValueError, regex='training mode'): predictions(nn.Linear(4, 2).train(), _dl)

# an empty dataloader raises
_empty = DataLoader(TensorDataset(torch.zeros(0, 4), torch.zeros(0, dtype=torch.long)), batch_size=8)
with ExceptionExpected(ValueError, regex='empty'): correct_vector(_model, _empty)

# anything callable on a batch works, numpy logits included (an ONNX session is one)
test_eq(predictions(lambda x: np.asarray(_model(x).detach()), _dl).tolist(), _p.tolist())

# a TorchScript module is scored like the eager model it was traced from, frozen ones included:
# freezing drops `training` altogether, which must not be read as a model left in training mode
_traced = torch.jit.trace(_model, torch.randn(2, 4))
_frozen = torch.jit.freeze(_traced)
test_eq(predictions(_traced, _dl).tolist(), _p.tolist())
test_eq(predictions(_frozen, _dl).tolist(), _p.tolist())
test_eq(correct_vector(_frozen, _dl).tolist(), correct_vector(_model, _dl).tolist())
assert not hasattr(_frozen, 'training')

# what it cannot trace, it says so about — never an AttributeError from the check above
with ExceptionExpected(ValueError, regex='RecursiveScriptModule'): peak_activation_bytes(_frozen, torch.randn(2, 4))

In [ ]:
# MACs, hand-counted: 8*8*8 outputs, each 3*3*3 multiply-accumulates
test_eq(macs(nn.Conv2d(3, 8, 3, padding=1).eval(), torch.randn(1, 3, 8, 8)), 13824)
test_eq(macs(nn.Linear(16, 4).eval(), torch.randn(1, 16)), 64)
test_eq(macs(nn.Linear(16, 4).eval(), torch.randn(1, 5, 16)), 5 * 64)   # a linear layer applied to 5 tokens
test_eq(macs(nn.Conv2d(4, 4, 3, groups=2).eval(), torch.randn(1, 4, 4, 4)), 288)   # 4*2*2 outputs, each 2*3*3

# one image, whatever the batch it is given
test_eq(macs(nn.Conv2d(3, 8, 3, padding=1).eval(), torch.randn(8, 3, 8, 8)), 13824)

# normalisation, activations and pooling are not multiply-accumulates
test_eq(macs(nn.Sequential(nn.BatchNorm2d(3), nn.ReLU(), nn.MaxPool2d(2)).eval(), torch.randn(1, 3, 8, 8)), 0)

_seq = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(inplace=True), nn.Conv2d(8, 4, 3, padding=1)).eval()
_img = torch.randn(1, 3, 8, 8)
test_eq(macs(_seq, _img), 13824 + 18432)
test_eq(params(_seq), 3 * 8 * 9 + 8 + 8 * 4 * 9 + 4)

# peak memory: at the second convolution its input (8*8*8*4 B) and its output (4*8*8*4 B) are live,
# and the image (3*8*8*4 B) is dead — the in-place ReLU wrote into the first convolution's buffer
test_eq(peak_activation_bytes(_seq, _img), 3072)


class _Res(nn.Module):
    def __init__(self): super().__init__(); self.c = nn.Conv2d(8, 8, 3, padding=1)
    def forward(self, x): return x + self.c(x)


# x, the convolution output and their sum are live at once
test_eq(peak_activation_bytes(_Res().eval(), torch.randn(1, 8, 8, 8)), 3 * 8 * 8 * 8 * 4)


class _Chunky(nn.Module):
    def forward(self, x):
        a, b = torch.chunk(x, 2, dim=1)
        return a + b


# a node whose output is a tuple counts both its tensors: the peak is at the second getitem, where the
# chunk pair (2 * 256 B) and the two halves it hands out (2 * 256 B) are live and x (512 B) has just died;
# at the addition only a, b and their sum are left, 768 B
test_eq(peak_activation_bytes(_Chunky().eval(), torch.randn(1, 8, 4, 4)), 1024)

In [ ]:
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import convert_fx, prepare_fx

_q = prepare_fx(copy.deepcopy(_seq), get_default_qconfig_mapping('x86'), (_img,))
with torch.no_grad(): _q(_img)
_q = convert_fx(_q)

# same topology, same multiply-accumulates; one byte per activation instead of four, so less memory
test_eq(macs(_q, _img), macs(_seq, _img))
assert peak_activation_bytes(_q, _img) < peak_activation_bytes(_seq, _img)

# a quantized module keeps its weight packed outside parameters(); its bias is not counted
test_eq(params(_q), 3 * 8 * 9 + 8 * 4 * 9)

In [ ]:
# a model measured in training mode raises, it never reports the batch it was given
with ExceptionExpected(ValueError, regex='training mode'): macs(nn.Conv2d(3, 8, 3), _img)
with ExceptionExpected(ValueError, regex='training mode'): peak_activation_bytes(nn.Conv2d(3, 8, 3), _img)


class _Branchy(nn.Module):
    def forward(self, x): return x * 2 if x.sum() > 0 else x


# a model FX cannot trace raises naming it, it never reports 0
with ExceptionExpected(ValueError, regex='_Branchy'): peak_activation_bytes(_Branchy().eval(), _img)